In [2]:
# packages
import elapid as ela
import os
import warnings
from pprint import pprint
import numpy as np
import pandas as pd
import itertools
import geopandas as gpd
import rasterio as rio
from rasterio import plot as rioplot
from sklearn import metrics
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import pathlib
import shutil
from maxent_insect_dir.spatial_kde import spatial_kernel_density
from scipy.spatial import cKDTree
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.inspection import permutation_importance

In [3]:
def spatial_thinning(gdf, min_distance):
    """
    Performs spatial thinning on a GeoDataFrame.

    Parameters:
    - gdf: GeoDataFrame with point geometries
    - min_distance: Minimum distance between points in meters

    Returns:
    - A GeoDataFrame with thinned points
    """
    # Extract coordinates
    coords = np.array([(geom.x, geom.y) for geom in gdf.geometry])

    # Build KDTree for efficient spatial queries
    tree = cKDTree(coords)

    # Initialize list to hold indices of points to keep
    keep_indices = []
    # Initialize a set to hold indices of points already considered
    considered = set()

    for idx in range(len(coords)):
        if idx in considered:
            continue
        # Query points within min_distance
        indices = tree.query_ball_point(coords[idx], r=min_distance)
        # Add current index to keep_indices
        keep_indices.append(idx)
        # Mark all indices within min_distance as considered
        considered.update(indices)

    # Return the thinned GeoDataFrame
    
    gdf = gdf.iloc[keep_indices]
    
    return gdf.reset_index(drop=True)

In [4]:
def spatial_weighted_background(presence, uniform_raster, weight=0, count=1e3):
    # weight 1 means everything from the bias, 0 mean everything is uniform
    n_point_biased =  int(count*weight) 
    n_point_uniform = int(count - n_point_biased)
    
    print(f'Number of point selected | uniformly: {n_point_uniform}  | with bias: {n_point_biased}\n\n')
    
    dir_path = 'temp'
    bias_raster = os.path.join(dir_path, 'bias.tif')
    if not os.path.exists(dir_path):
        # Create the directory
        os.makedirs(dir_path)
        print(f"Directory '{dir_path}' created.")
        
    # create the bias using presence point
    spatial_kernel_density(
        points=presence,
        radius=160000,
        output_path=bias_raster,
        output_pixel_size = 16000,
        output_driver='GTiff',
        scaled=True
    )


    point_sampled_uniformly = gpd.GeoDataFrame(geometry=[], crs="EPSG:5070")   
    point_sampled_with_bias = gpd.GeoDataFrame(geometry=[], crs="EPSG:5070")
    
    if weight == 0:
        point_sampled_uniformly = ela.sample_raster(uniform_raster, n_point_uniform).to_crs(epsg=5070) 
    elif weight == 1:
        point_sampled_with_bias =  ela.sample_bias_file(bias_raster, n_point_biased).to_crs(epsg=5070) 
    else:
        point_sampled_uniformly = ela.sample_raster(uniform_raster, n_point_uniform).to_crs(epsg=5070) 
        point_sampled_with_bias = ela.sample_bias_file(bias_raster, n_point_biased).to_crs(epsg=5070) 
    
    shutil.rmtree(dir_path)
    
    merged = pd.concat(
        [point_sampled_uniformly.geometry, point_sampled_with_bias.geometry], 
        ignore_index=True
    )
    
    merged = gpd.GeoSeries(merged, crs="EPSG:5070")
    
    return gpd.GeoDataFrame(geometry=merged, crs="EPSG:5070")

In [5]:
def plot_correlation_matrix(corr_matrix, title, save_path=None):
    """Function to plot the correlation matrix using seaborn heatmap"""
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
    plt.title(title)
    plt.show()
    if save_path is not None:
        plt.savefig(save_path)


def remove_highly_correlated(df, threshold=0.9):
    """
    Remove highly correlated features from the dataframe.
    
    :param df: Input DataFrame with continuous variables
    :param threshold: Correlation threshold above which to remove variables
    :return: DataFrame with removed highly correlated features
    """
    corr_matrix = df.corr()  # Compute correlation matrix for continuous variables
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))

    # Find features with correlation greater than the threshold
    to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column].abs() > threshold)]
    
    print(f"Columns to drop (correlation > {threshold}): {to_drop}")
    return df.drop(to_drop, axis=1)


def remove_uncorrelated_columns(df, threshold=0.9, save_path=None, tag=None, plot=False):
    # Remove highly correlated features
    df_filtered = remove_highly_correlated(df, threshold=threshold)
    
    # Compute and plot correlation matrix
    corr_matrix = df.corr()
    corr_matrix_uncorr = df_filtered.corr()
    if plot:
        plot_correlation_matrix(corr_matrix, "Correlation Matrix Before Removing Highly Correlated Features")
        plot_correlation_matrix(corr_matrix_uncorr, "Correlation Matrix After Removing Highly Correlated Features")    
    
    return df_filtered

In [6]:
def plot_sample_weights(df_presence, df_background, save_path=None):
    
    _, ax = plt.subplots(1,2,figsize=(15,3))
    df_presence.plot(column='SampleWeight', legend=True, cmap='YlOrBr', ax=ax[0])
    df_background.plot(column='SampleWeight', legend=True, cmap='YlOrBr', ax=ax[1])
    
    ax[0].set_title('presence sampleweight')
    ax[0].set_xlabel("Longitude")
    ax[0].set_ylabel("Latitude")

    ax[1].set_xlabel("Longitude")
    ax[1].set_ylabel("Latitude")
    ax[1].set_title('background sampleweight')
    
    plt.show()
    if save_path is not None:
        plt.savefig(save_path)

In [7]:
def get_train_test_split(df, use_sample_weights=False, save_path=None, grid_size=0.05, plot=False):
    
    if not df.crs.is_geographic:
        grid_size = 50000
    
    print(f'Using the grid_size : {grid_size}..\n\n')
    
    train, test = ela.checkerboard_split(df, grid_size=grid_size)
    
    # we'll re-merge them for plotting purposes
    train['split'] = 'train'
    test['split'] = 'test'
    
    # Create GeoDataFrames for train and test
    train_gdf = gpd.GeoDataFrame(train, geometry='geometry', crs=df.crs)
    test_gdf = gpd.GeoDataFrame(test, geometry='geometry', crs=df.crs)
    
    # split train-test
    columns = ['class', 'split', 'geometry']
    if use_sample_weights:
        columns.append('SampleWeight')
        
    x_train = train.drop(columns=columns)
    y_train = train['class']
    x_test = test.drop(columns=columns)
    y_test = test['class']
    
    if plot:
        checker = ela.stack_geodataframes(train, test)
        checker.plot(column='split', markersize=0.75, legend=True)
        plt.show()
        if save_path is not None:
            plt.savefig(save_path)

    if use_sample_weights:
        sample_weight_train = train['SampleWeight']
        sample_weight_test = test['SampleWeight']    
        return x_train, y_train, x_test, y_test, sample_weight_train, sample_weight_test, train_gdf, test_gdf
    else:
        return x_train, y_train, x_test, y_test, train_gdf, test_gdf

In [ ]:
# I am taking input here

vector = pathlib.Path('/data/Maxent/data/raw/Species Data/Guna/Clean_Data/Brown_Marmorated_Stink_Bug -  (Halyomorpha_halys) (2007-2023).gpkg')
raster_path = pathlib.Path('/data/Maxent/data/raw/Bioclim_worldclim_average_1970_2000/wc2.1_5m_bio/Clipped/In_tiff_format')

raster_names = [
    'bio1', 'bio2', 'bio3', 'bio4', 'bio5', 'bio6', 'bio7', 'bio8', 'bio9', 'bio10',
    'bio11', 'bio12', 'bio13', 'bio14', 'bio15', 'bio16', 'bio17', 'bio18', 'bio19',
]

rasters = [os.path.join(raster_path, f'{i}.tif') for i in raster_names]

In [ ]:
# read the presence data, draw background point samples
presence_ = gpd.read_file(vector)[['label', 'geometry']]
presence = spatial_thinning(presence_, 30000)

presence = ela.annotate(
    presence,
    rasters,
    labels = raster_names,
    drop_na=True
)

presence['SampleWeight'] = ela.distance_weights(presence, n_neighbors=-1, center=None)



background = spatial_weighted_background(
    presence=presence,
    uniform_raster=rasters[0],
    weight=0.75,
    count=10000
)

background = ela.annotate(
    background,
    rasters,
    labels=raster_names,
    drop_na=True
)

background['SampleWeight'] = ela.distance_weights(background, n_neighbors=-1, center=None)


fig, ax = plt.subplots(1,3, figsize=(15,5))
presence_.plot(ax=ax[0], alpha=0.1)
ax[0].set_title('Presence')
presence.plot(ax=ax[1], alpha=0.1)
ax[1].set_title('Presence spatially thinned')
background.plot(ax=ax[2], alpha=0.1)
ax[2].set_title('Background')

print(f'Presence original | {presence_.shape} | Presence spatially thinned | {presence.shape}' )

In [ ]:
ncols = 4
nrows = len(raster_names)//ncols

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols,3*nrows))
pair_colors = ['blue', 'green']
for name, ax in zip(raster_names, axs.ravel()):
    pvar = presence[name]
    bvar = background[name]
    ax.hist(
        [pvar, bvar],
        density=True,
        alpha=0.7,
        label=['presence', 'background'],
        color=pair_colors,
    )
    ax.set_title(name)
    
handles, lbls = ax.get_legend_handles_labels()
fig.legend(handles, lbls, loc=(0.6, 0.9))
plt.tight_layout()

In [ ]:
plot_sample_weights(presence, background, save_path=None)

In [14]:
# merge datasets and read the covariates at each point location
annotated = ela.stack_geodataframes(presence, background, add_class_label=True)

In [15]:
# raster_files_mapping = dict(zip(annotated.columns[2:], rasters))

In [16]:
base_cols = ['geometry', 'class', 'SampleWeight']

In [ ]:
cols_to_keep = remove_uncorrelated_columns(df=annotated[raster_names], plot=True, threshold=0.8).columns
annotated = annotated[base_cols + list(cols_to_keep)]

In [ ]:
use_sample_weights = True

if use_sample_weights == True:
    x_train, y_train, x_test, y_test, sample_weight_train, sample_weight_test, train_gdf, test_gdf = get_train_test_split(
        annotated,
        use_sample_weights,
        plot=True
    )
else:
    x_train, y_train, x_test, y_test, train_gdf, test_gdf = get_train_test_split(
        annotated,
        plot=True
    )

In [19]:
cat_indices = None
# train the model
model = ela.MaxentModel(transform='cloglog', beta_multiplier=2.0)
if use_sample_weights:
    model.fit(x_train, y_train, sample_weight=sample_weight_train, categorical=cat_indices)
else:
    model.fit(x_train, y_train, categorical=cat_indices)

In [ ]:
# Define the custom scorer with pos_label=1
scorer = make_scorer(roc_auc_score, needs_proba=True)

# Ensure classes_ is a NumPy array
model.classes_ = np.array(model.classes_)

# Calculate permutation importance
result = permutation_importance(model, x_test, y_test, n_repeats=10, random_state=42, scoring=scorer)


# Store feature importance in metadata
feature_importance = pd.Series(result.importances_mean, index=x_test.columns)

plot = True

if plot:
    # Optionally, display or plot the feature importance
    print("Feature importance calculated using permutation importance:\n", feature_importance)
    # Sort and prepare the data
    feature_importance = feature_importance.sort_values(ascending=False)
    # Plot with enhancements
    plt.figure(figsize=(12, 6))  # Adjust figure size
    feature_importance.plot(kind='bar', title='Permutation Importance')
    plt.ylabel('Mean Decrease in AUC')
    plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels
    plt.tight_layout()  # Adjust layout
    plt.show()

In [ ]:
y_pred = model.predict(x_test)
y_pred_train = model.predict(x_train)

Unweighted_checkerboard_AUC_score_on_test = metrics.roc_auc_score(y_test, y_pred)

if use_sample_weights:
    weighted_checkerboard_AUC_score_on_test = metrics.roc_auc_score(y_test, y_pred, sample_weight=sample_weight_test)
    
print(f'Unweighted_checkerboard_AUC_score_on_test | {round(Unweighted_checkerboard_AUC_score_on_test,3)}')
print(f'Weighted_checkerboard_AUC_score_on_test | {round(weighted_checkerboard_AUC_score_on_test,3)}')

In [24]:
raster_files_mapping = dict(zip(raster_names, rasters))

In [26]:
updated_rasters = [raster_files_mapping[i] for i in cols_to_keep]

In [ ]:
# partial dependence plots show how model predictions vary across the range of co-variates
fig, ax = model.partial_dependence_plot(x_train, labels=cols_to_keep, dpi=100)

In [28]:
# write the model predictions to disk
output_raster = 'demo-maxent-predictions.tif'
ela.apply_model_to_rasters(model, updated_rasters, output_raster, quiet=True)

# and read into memory
with rio.open(output_raster, 'r') as src:
    pred = src.read(1, masked=True)

In [ ]:
# plot the suitability predictions
fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=100)
plot = ax.imshow(pred, vmin=0, vmax=1, cmap='viridis')
ax.set_title('$Ariolimax buttoni$ suitability')
ax.set_xticks([])
ax.set_yticks([])
cbar = plt.colorbar(plot, ax=ax, label="relative occurrence probability", pad=0.04)
plt.tight_layout()

In [30]:
from maxent_insect_dir.Continuous_boyce_index import *

In [ ]:
print("Step 8a: Calculating continuous Boyce index for train sets....\n\n")
CBI_train = continuous_boyce_index(
    yobs=model.predict(x_train[y_train == 1]),  # prediction on presence
    ypred=model.predict(x_train[y_train == 0]),  # prediction on background
    to_plot=plot
)


print("Step 8b: Calculating continuous Boyce index for test sets....\n\n")
CBI_test = continuous_boyce_index(
    yobs=model.predict(x_test[y_test == 1]),  # prediction on presence
    ypred=model.predict(x_test[y_test == 0]),  # prediction on background
    to_plot=plot
)